<a href="https://colab.research.google.com/github/Lattemelia/Portforio/blob/main/demand%20forecasting1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [5]:
marged_df

,id,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,188533,Land,Rover LR2 Base,2015,98000,Gasoline,240.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Beige,None reported,Yes,NaN
1,188534,Land,Rover Defender SE,2020,9142,Hybrid,395.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,8-Speed A/T,Silver,Black,None reported,Yes,NaN
2,188535,Ford,Expedition Limited,2022,28121,Gasoline,3.5L V6 24V PDI DOHC Twin Turbo,10-Speed Automatic,White,Ebony,None reported,NaN,NaN
3,188536,Audi,A6 2.0T Sport,2016,61258,Gasoline,2.0 Liter TFSI,Automatic,Silician Yellow,Black,None reported,NaN,NaN
4,188537,Audi,A6 2.0T Premium Plus,2018,59000,Gasoline,252.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,A/T,Gray,Black,None reported,Yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,188528,Cadillac,Escalade ESV Platinum,2017,49000,Gasoline,420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,White,Beige,None reported,Yes,27500.0
188529,188529,Mercedes-Benz,AMG C 43 AMG C 43 4MATIC,2018,28600,Gasoline,385.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,30000.0
188530,188530,Mercedes-Benz,AMG GLC 63 Base 4MATIC,2021,13650,Gasoline,469.0HP 4.0L 8 Cylinder Engine Gasoline Fuel,7-Speed A/T,White,Black,None reported,Yes,86900.0
188531,188531,Audi,S5 3.0T Prestige,2022,13895,Gasoline,3.0L,1-Speed Automatic,Daytona Gray Pearl Effect,Black,None reported,NaN,84900.0


In [6]:
marged_df.isnull().sum()

,0
id,0
brand,0
model,0
model_year,0
milage,0
fuel_type,8466
engine,0
transmission,0
ext_col,0
int_col,0


In [7]:
#エンジン名のみ抽出
root_regex = r"(?:^\d+\.\d+HP\s+)?([^\s]+)(?:\s+(.+))?$"

# 元の engine 列から直接、一撃で2カラムに代入
marged_df[["engine_displacement", "engine_type"]] = marged_df["engine"].str.extract(root_regex)


In [8]:
marged_df[marged_df["engine_type"].isnull()]

,id,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,engine_displacement,engine_type
179,188712,Tesla,Model S Long Range Plus,2022,24114,NaN,Electric,Automatic,Midnight Silver Metallic,Jet Black,At least 1 accident or damage reported,NaN,NaN,Electric,NaN
592,189125,Land,Rover Range Rover Evoque SE,2018,2908,Gasoline,V8,Automatic,Santorini Black Metallic,Pimento Red w/Ebony,None reported,NaN,NaN,V8,NaN
687,189220,Tesla,Model 3 Long Range,2018,27224,NaN,Electric,Automatic,Silver,–,None reported,NaN,NaN,Electric,NaN
704,189237,Dodge,Challenger SRT8 392,2010,33700,–,–,A/T,White,Gray,None reported,Yes,NaN,–,NaN
714,189247,Porsche,Taycan 4S,2018,107714,NaN,Electric,2-Speed Automatic,Gentian Blue Metallic,Black,None reported,NaN,NaN,Electric,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187964,187964,Mercedes-Benz,C-Class 4MATIC Sedan,1993,22703,Gasoline,–,A/T,Gray,–,None reported,Yes,220000.0,–,NaN
188189,188189,Chrysler,300 Touring,1994,79785,–,–,A/T,Green,Beige,At least 1 accident or damage reported,Yes,9499.0,–,NaN
188337,188337,Ford,Mustang Mach-E GT,2022,25361,NaN,Electric,1-Speed Automatic,White,Black Onyx,None reported,Yes,51074.0,Electric,NaN
188362,188362,Volkswagen,e-Golf SE,2021,22185,NaN,Electric,1-Speed Automatic,Brilliant Silver Metallic,Graphite w/Gun Metal,None reported,NaN,34999.0,Electric,NaN


In [9]:
displacement_counts = marged_df.loc[marged_df["engine_type"].isnull(), "engine_displacement"].value_counts()

# 結果をすべて表示したい場合は、以前設定したように print するか、そのまま出力します
print(displacement_counts)

engine_displacement
–           1542
Electric     808
V6           166
I4           103
3.0L          62
V8            48
4.0L          35
2.5L           9
Name: count, dtype: int64


In [10]:
marged_df = marged_df.reset_index(drop=True)

#EVフラグの作成
marged_df["is_electric"] = np.where(
    marged_df["engine_displacement"] == "Electric", 1, 0
)

#型式の誤侵入を修正
invalid_displacement_mask = marged_df["engine_displacement"].isin(
    ["V6", "I4", "V8"]
)

#インデックスを振り直したため、エラーにならずに代入できるようになります
marged_df.loc[invalid_displacement_mask, "engine_type"] = marged_df.loc[
    invalid_displacement_mask, "engine_displacement"
]

#移動させたので、排気量側は一旦 NaN（空）にする
marged_df.loc[invalid_displacement_mask, "engine_displacement"] = np.nan


#排気量を「純粋な数値」に変換する
marged_df["engine_displacement_num"] = pd.to_numeric(
    marged_df["engine_displacement"].str.replace("L", "", regex=False),
    errors="coerce",
)

In [11]:
marged_df.loc[marged_df["engine_displacement_num"].isnull(), "engine_displacement"].value_counts()

,count
engine_displacement,
Electric,8889
–,1542
Dual,226
Intercooled,130
Standard,53
Battery,18
111.2Ah,9


In [25]:
marged_df2 =  marged_df[marged_df["engine_displacement_num"].isnull()]

# 1. 除外したいキーワードのリストを作成（表記揺れにも対応）
exclude_keywords = ["–", "-", "Electric"]

# 2. 条件を作成する
# 条件A: engine_displacement が NaN（欠損値）
cond_not_null = marged_df2["engine_displacement"].notnull()

# 条件B: engine_displacement が 除外リストに含まれて「いない」
cond_not_in_list = ~marged_df2["engine_displacement"].isin(exclude_keywords)

marged_df3 = marged_df2[cond_not_null & cond_not_in_list]

marged_df4 = marged_df3[~marged_df3["engine_displacement"].isin(["Electric","-"])]



In [29]:
marged_df4.groupby("engine_displacement")["engine"].value_counts()

engine_displacement  engine                                          
111.2Ah              111.2Ah / FR 70kW / RR 160kW (697V)                   9
Battery              Battery Electric                                     18
Dual                 Dual Motor - Standard                               200
                     Dual AC Electric Motors                              26
Intercooled          Intercooled Turbo Premium Unleaded I-4 2.0 L/122     66
                     Intercooled Turbo Diesel V-8 6.7 L/406               64
Standard             Standard Range Battery                               53
Name: count, dtype: int64

In [33]:
marged_df.loc[marged_df["engine_displacement"].isin(["111.2Ah","Battery","Dual","Standard"]),"engine_displacement"] == "Electric"

,engine_displacement
1662,False
6142,False
6742,False
6978,False
7072,False
...,...
307092,False
307689,False
308207,False
309337,False
